In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # JobFlow AI — Validação Final
# MAGIC
# MAGIC Este notebook valida se os principais componentes do projeto foram criados:
# MAGIC
# MAGIC - tabelas Bronze, Silver e Gold;
# MAGIC - ingestão e auditoria;
# MAGIC - processamento de texto não estruturado;
# MAGIC - matching perfil x vagas;
# MAGIC - tabelas transacionais do app;
# MAGIC - ações do mini copiloto;
# MAGIC - dados necessários para o frontend simulado.

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql import types as T

# COMMAND ----------

CATALOG = "workspace"
SCHEMA = "jobflow_ai"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

DEMO_USER_ID = "demo_user_001"

print("Validação final iniciada.")
print(f"Schema ativo: {CATALOG}.{SCHEMA}")
print(f"Usuário demo: {DEMO_USER_ID}")

# COMMAND ----------

expected_tables = [
    "ingestion_runs",
    "bronze_remoteok_jobs",
    "silver_remoteok_jobs",
    "gold_job_postings",
    "gold_job_description_chunks",
    "skills_catalog",
    "gold_job_requirement_sentences",
    "gold_job_skill_matches",
    "demo_user_profiles",
    "demo_user_profile_skills",
    "gold_job_match_scores",
    "app_users",
    "app_profiles",
    "app_skills",
    "app_job_postings",
    "app_saved_jobs",
    "app_applications",
    "app_interview_notes",
    "app_contacts",
]

# COMMAND ----------

def full_table_name(table_name):
    return f"{CATALOG}.{SCHEMA}.{table_name}"


def table_exists(table_name):
    try:
        spark.table(full_table_name(table_name)).limit(1).count()
        return True
    except Exception:
        return False


def safe_count(table_name):
    try:
        return spark.table(full_table_name(table_name)).count()
    except Exception:
        return None


validation_rows = []

for table_name in expected_tables:
    exists = table_exists(table_name)
    row_count = safe_count(table_name) if exists else None

    validation_rows.append(
        {
            "area": "tabela",
            "check_name": f"existe_{table_name}",
            "target": full_table_name(table_name),
            "status": "OK" if exists else "FALHOU",
            "value": str(row_count) if row_count is not None else "tabela não encontrada",
        }
    )

validation_schema = T.StructType(
    [
        T.StructField("area", T.StringType(), True),
        T.StructField("check_name", T.StringType(), True),
        T.StructField("target", T.StringType(), True),
        T.StructField("status", T.StringType(), True),
        T.StructField("value", T.StringType(), True),
    ]
)

validation_df = spark.createDataFrame(validation_rows, validation_schema)

display(validation_df.orderBy("status", "target"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Contagens principais

# COMMAND ----------

main_counts = []

count_targets = {
    "bronze_records": "bronze_remoteok_jobs",
    "silver_records": "silver_remoteok_jobs",
    "gold_jobs": "gold_job_postings",
    "description_chunks": "gold_job_description_chunks",
    "requirement_sentences": "gold_job_requirement_sentences",
    "skill_matches": "gold_job_skill_matches",
    "match_scores": "gold_job_match_scores",
    "app_job_postings": "app_job_postings",
    "saved_jobs": "app_saved_jobs",
    "applications": "app_applications",
    "interview_notes": "app_interview_notes",
}

for metric_name, table_name in count_targets.items():
    main_counts.append(
        {
            "metric": metric_name,
            "table_name": full_table_name(table_name),
            "row_count": safe_count(table_name),
        }
    )

main_counts_df = spark.createDataFrame(main_counts)

display(main_counts_df.orderBy("metric"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Auditoria de ingestão

# COMMAND ----------

display(
    spark.table(full_table_name("ingestion_runs"))
    .orderBy(F.col("started_at").desc())
)

# COMMAND ----------

display(
    spark.table(full_table_name("ingestion_runs"))
    .groupBy("source", "status")
    .agg(
        F.count("*").alias("runs"),
        F.sum("records_raw").alias("total_records_raw"),
        F.sum("records_bronze").alias("total_records_bronze"),
    )
    .orderBy("source", "status")
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Top vagas recomendadas

# COMMAND ----------
top_recommendations_df = (
    spark.table(full_table_name("gold_job_match_scores")).alias("m")
    .where(F.col("m.user_id") == DEMO_USER_ID)
    .join(
        spark.table(full_table_name("app_job_postings")).alias("j"),
        F.col("m.job_id") == F.col("j.job_id"),
        "left",
    )
    .select(
        F.col("m.user_id"),
        F.col("m.job_id"),
        F.col("j.job_title").alias("title"),
        F.col("j.company_name").alias("company"),
        F.col("j.job_url"),
        F.col("m.match_score"),
    )
    .orderBy(F.col("m.match_score").desc())
    .limit(10)
)

display(top_recommendations_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Ações gravadas pelo copiloto

# COMMAND ----------

display(
    spark.table(full_table_name("app_saved_jobs"))
    .where(F.col("user_id") == DEMO_USER_ID)
    .orderBy(F.col("updated_at").desc())
)

display(
    spark.table(full_table_name("app_applications"))
    .where(F.col("user_id") == DEMO_USER_ID)
    .orderBy(F.col("updated_at").desc())
)

display(
    spark.table(full_table_name("app_interview_notes"))
    .where(F.col("user_id") == DEMO_USER_ID)
    .orderBy(F.col("created_at").desc())
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Validação dos requisitos do projeto

# COMMAND ----------

def metric_count(table_name):
    value = safe_count(table_name)
    return 0 if value is None else value


bronze_count = metric_count("bronze_remoteok_jobs")
silver_count = metric_count("silver_remoteok_jobs")
gold_count = metric_count("gold_job_postings")
chunks_count = metric_count("gold_job_description_chunks")
match_count = metric_count("gold_job_match_scores")
saved_count = metric_count("app_saved_jobs")
applications_count = metric_count("app_applications")
notes_count = metric_count("app_interview_notes")

requirement_checks = [
    {
        "requirement": "Spark data pipeline",
        "evidence": f"bronze={bronze_count}, silver={silver_count}, gold={gold_count}",
        "status": "OK" if bronze_count > 0 and silver_count > 0 and gold_count > 0 else "FALHOU",
    },
    {
        "requirement": "API externa com auditoria",
        "evidence": "Tabela ingestion_runs registra tentativas de ingestão e fallback manual",
        "status": "OK" if metric_count("ingestion_runs") > 0 else "FALHOU",
    },
    {
        "requirement": "Processamento de dados não estruturados",
        "evidence": f"description_chunks={chunks_count}",
        "status": "OK" if chunks_count > 0 else "FALHOU",
    },
    {
        "requirement": "Matching perfil x vagas",
        "evidence": f"match_scores={match_count}",
        "status": "OK" if match_count > 0 else "FALHOU",
    },
    {
        "requirement": "Ferramentas de escrita do agente",
        "evidence": f"saved_jobs={saved_count}, applications={applications_count}, notes={notes_count}",
        "status": "OK" if saved_count > 0 and applications_count > 0 and notes_count > 0 else "FALHOU",
    },
    {
        "requirement": "Frontend ou interface do app",
        "evidence": "frontend_simulado.py.ipynb mostra métricas, perfil, vagas recomendadas, vagas salvas e aplicações",
        "status": "OK",
    },
]

requirement_checks_df = spark.createDataFrame(requirement_checks)

display(requirement_checks_df.orderBy("requirement"))

# COMMAND ----------

ok_count = requirement_checks_df.where(F.col("status") == "OK").count()
total_count = requirement_checks_df.count()

displayHTML(f"""
<div style="
    padding: 24px;
    border-radius: 16px;
    background: linear-gradient(135deg, #064e3b, #022c22);
    color: white;
    font-family: Arial, sans-serif;
">
    <h1 style="margin-bottom: 8px;">Validação Final — JobFlow AI</h1>
    <p style="font-size: 16px; margin-top: 0;">
        Requisitos validados: <strong>{ok_count}/{total_count}</strong>
    </p>
    <div style="margin-top: 18px; font-size: 15px;">
        <p><strong>Pipeline:</strong> Bronze, Silver e Gold criados.</p>
        <p><strong>Texto não estruturado:</strong> descrições de vagas processadas em chunks.</p>
        <p><strong>Matching:</strong> vagas ranqueadas para o usuário demo.</p>
        <p><strong>Agente:</strong> ferramentas de leitura e escrita validadas.</p>
        <p><strong>Frontend:</strong> interface simulada criada em notebook.</p>
    </div>
</div>
""")

# COMMAND ----------

print()
print("=" * 70)
print("RESULTADO: VALIDAÇÃO FINAL CONCLUÍDA")
print("=" * 70)
print(f"Requisitos validados: {ok_count}/{total_count}")
print()
print("Componentes verificados:")
print("- tabelas Bronze, Silver e Gold")
print("- auditoria de ingestão")
print("- chunks de descrições de vagas")
print("- extração de requisitos e skills")
print("- matching perfil x vagas")
print("- tabelas transacionais do app")
print("- ações gravadas pelo copiloto")
print("- frontend simulado")
print("=" * 70)